# TV24 Newsroom — render the anchor video (InfiniteTalk, free Kaggle GPU)

Turns a bulletin's voice tracks into talking-anchor videos, one MP4 per segment.

**Before you press Run All**
1. On your Mac: `cd server && python -m app.render prepare` → makes `server/renders/<id>.zip`
2. Kaggle → *Datasets* → *New Dataset* → upload that zip (Kaggle unzips it).
3. In this notebook: *Add Input* → your dataset. *Settings* → **Accelerator: GPU T4 x2**, **Internet: On**.
4. *Run All*. When it finishes, open the *Output* panel and download `videos_<id>.zip`.
5. On your Mac: `python -m app.render publish ~/Downloads/videos_<id>.zip`, then commit + push.

**Time:** a free T4 is slow — expect roughly 10–25 minutes per 30 seconds of speech at the default
settings. The first run also downloads about 30 GB of model weights (~10–15 min). Test with one segment
first by setting `ONLY = ["01_story"]` below.

Model: [InfiniteTalk](https://github.com/MeiGen-AI/InfiniteTalk) (Apache-2.0) on Wan 2.1, run in its
fp8 low-memory mode so it fits the free GPU and 30 GB of RAM.

In [ ]:
# ---- settings ----
STEPS = 30                  # 40 = best quality (slowest), 20 = faster, softer
SIZE = "infinitetalk-480"   # 480p; "infinitetalk-720" needs a bigger GPU than the free T4
ONLY = None                 # e.g. ["01_story"] to try a single segment first

In [ ]:
# ---- find the render job (attached dataset) ----
import glob, json, os, zipfile, shutil
cands = glob.glob('/kaggle/input/**/bulletin.json', recursive=True)
if not cands:  # dataset kept as a .zip
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        zipfile.ZipFile(z).extractall('/tmp/job')
    cands = glob.glob('/tmp/job/**/bulletin.json', recursive=True)
assert cands, "No render job found. Attach the zip from `python -m app.render prepare` as a dataset (Add Input)."
JOB = os.path.dirname(cands[0])
B = json.load(open(cands[0], encoding='utf-8'))
SEGS = B['render']['segments']
print('Bulletin', B['id'], '-', len(SEGS), 'segments')
for s in SEGS:
    print(' ', s['key'], '|', s['headline'])

In [ ]:
# ---- install InfiniteTalk (fresh process, so no kernel restart needed) ----
!git clone -q --depth 1 https://github.com/MeiGen-AI/InfiniteTalk /tmp/InfiniteTalk
!pip install -q torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 xformers==0.0.28 --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r /tmp/InfiniteTalk/requirements.txt "misaki[en]" librosa pyloudnorm einops soundfile "huggingface_hub>=0.24"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# ---- download weights (~30 GB, fp8 low-memory set only) ----
from huggingface_hub import snapshot_download, hf_hub_download
W = '/tmp/weights'
snapshot_download('Wan-AI/Wan2.1-I2V-14B-480P', local_dir=f'{W}/Wan2.1-I2V-14B-480P',
                  ignore_patterns=['diffusion_pytorch_model*', 'models_t5_umt5-xxl-enc-bf16.pth', 'examples/*', 'assets/*'])
snapshot_download('TencentGameMate/chinese-wav2vec2-base', local_dir=f'{W}/chinese-wav2vec2-base')
hf_hub_download('TencentGameMate/chinese-wav2vec2-base', 'model.safetensors', revision='refs/pr/1',
                local_dir=f'{W}/chinese-wav2vec2-base')
snapshot_download('MeiGen-AI/InfiniteTalk', local_dir=f'{W}/InfiniteTalk',
                  allow_patterns=['single/*', 'quant_models/infinitetalk_single_fp8*', 'quant_models/t5_fp8*', 'quant_models/t5_map_fp8*'])
!du -sh /tmp/weights/*

In [ ]:
# ---- render every segment ----
import math, subprocess, time, librosa
OUT = f"/kaggle/working/videos_{B['id']}"
os.makedirs(OUT, exist_ok=True)
shutil.copy(cands[0], f'{OUT}/bulletin.json')
frame = os.path.join(JOB, B['render']['frame'])

for s in SEGS:
    key = s['key']
    if ONLY and key not in ONLY:
        continue
    if os.path.exists(f'{OUT}/{key}.mp4'):
        print('skip (done)', key); continue
    audio = os.path.join(JOB, s['audio'])
    dur = librosa.get_duration(path=audio)
    job = {'prompt': B['render']['prompt'], 'cond_video': frame, 'cond_audio': {'person1': audio}}
    jp = f'/tmp/{key}.json'
    json.dump(job, open(jp, 'w'), ensure_ascii=False)
    cmd = ['python', 'generate_infinitetalk.py',
           '--ckpt_dir', f'{W}/Wan2.1-I2V-14B-480P',
           '--wav2vec_dir', f'{W}/chinese-wav2vec2-base',
           '--infinitetalk_dir', f'{W}/InfiniteTalk/single/infinitetalk.safetensors',
           '--quant', 'fp8',
           '--quant_dir', f'{W}/InfiniteTalk/quant_models/infinitetalk_single_fp8.safetensors',
           '--input_json', jp, '--size', SIZE, '--sample_steps', str(STEPS),
           '--mode', 'streaming', '--motion_frame', '9', '--num_persistent_param_in_dit', '0',
           '--max_frame_num', str(math.ceil(dur * 25) + 25),
           '--save_file', f'{OUT}/{key}']
    print(f'▶ {key}  ({dur:.0f}s of speech)  {s["headline"]}')
    t = time.time()
    r = subprocess.run(cmd, cwd='/tmp/InfiniteTalk')
    ok = r.returncode == 0 and os.path.exists(f'{OUT}/{key}.mp4')
    print(('  ✓' if ok else '  ✗ failed'), f'{(time.time() - t) / 60:.1f} min')

In [ ]:
# ---- package for download ----
done = sorted(glob.glob(f'{OUT}/*.mp4'))
print(len(done), 'of', len(SEGS), 'videos rendered')
zip_path = shutil.make_archive(OUT, 'zip', OUT)
print('Download from the Output panel:', zip_path)